# Topic 1: Shuffle Internals

> **Phase 4 → Week 7 → Topic 1**
>
> Prerequisites: Week 6 complete (Memory Model, Executor Config, GC, Spill)

---

## What is a Shuffle?

A shuffle is Spark's way of redistributing data across partitions — moving data from where it IS to where it NEEDS to BE for the next operation.

```
BEFORE SHUFFLE (data spread arbitrarily):   AFTER SHUFFLE (grouped by key):

Partition 0: [A:1, B:5, C:2]               Partition 0: [A:1, A:9, A:3]
Partition 1: [A:9, C:7, D:4]    ──────→    Partition 1: [B:5, B:2]
Partition 2: [A:3, B:2, D:8]               Partition 2: [C:2, C:7]
                                            Partition 3: [D:4, D:8]

All A's together, all B's together, etc.
This is required for: groupBy, join (SMJ), distinct, orderBy, repartition
```

---

## Sort Merge Shuffle (Default since Spark 1.2)

```
SHUFFLE WRITE PHASE (map side):
  1. Each task partitions its output by hash(key) % num_partitions
  2. Sorts each partition's data
  3. Writes sorted data to local disk: one shuffle file per task
     (indexed file + data file)

SHUFFLE READ PHASE (reduce side):
  1. Each downstream task fetches its partition's data from ALL upstream tasks
     (over the network)
  2. Merges the sorted streams (merge-sort)
  3. Processes the merged stream

Why disk? Upstream tasks must FINISH writing before downstream can read.
Disk acts as the buffer between the two stages.
```

---

## Shuffle Cost Breakdown

```
Stage 1 (write): CPU (sort) + Disk write
                         ↓
                    Barrier point
                    (ALL map tasks must finish)
                         ↓
Stage 2 (read):  Network I/O + Disk read + CPU (merge) 

Shuffle is the most expensive operation in Spark because:
  1. Forces a stage boundary (barrier point)
  2. Disk I/O (write + read)
  3. Network I/O (data moves between nodes)
  4. CPU overhead (sort + merge)
```

---

## Interview Cheat Sheet

**Q: What is a shuffle in Spark? Why is it expensive?**
> A shuffle redistributes data across partitions — all rows with the same key must end up in the same partition. It's expensive because: (1) it creates a stage barrier — all upstream tasks must finish before any downstream task starts; (2) it involves disk I/O (writing sorted shuffle files); (3) it involves network I/O (data fetched across nodes); (4) it has CPU overhead (sorting and merging). Every groupBy, join (non-broadcast), distinct, and orderBy triggers a shuffle.

**Q: What triggers a shuffle in Spark?**
> Wide transformations that require co-locating rows with the same key: `groupBy`, `agg`, `join` (SortMergeJoin), `distinct`, `repartition`, `orderBy`, `coalesce` (sometimes), `intersection`, `subtract`. Narrow transformations (map, filter, union) do NOT trigger shuffles.

**Q: How many shuffle files does Spark create?**
> With Sort Merge Shuffle: `M × R` — M mapper tasks × R reducer partitions. For 100 mappers and 200 reduce partitions: 20,000 files. This is why default `shuffle.partitions=200` can cause small file problems. Spark 3.2+ with "push-based shuffle" reduces this.

**Q: What is a shuffle spill?**
> When the sort buffer for shuffle write doesn't fit in execution memory, Spark spills partial sorted data to disk, then merges all spill files at the end. The merged spill + final sort = more I/O. Visible in UI as Spill (Memory/Disk). Fix: more partitions (smaller sort buffers) or more executor memory.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("Week7 - Shuffle Internals") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready — AQE disabled to observe raw shuffle behavior")

## Part 1: Identifying Shuffles in the DAG

In [ ]:
orders   = spark.read.csv("/workspace/week4/data/orders.csv",   header=True, inferSchema=True)
customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
products  = spark.read.csv("/workspace/week4/data/products.csv",  header=True, inferSchema=True)

# Every "Exchange" node in explain() = one shuffle
# Count how many shuffles each operation triggers

def count_exchanges(df, label):
    plan = df._jdf.queryExecution().executedPlan().toString()
    exchanges = plan.count("Exchange")
    print(f"{label}: {exchanges} shuffle(s)")

count_exchanges(orders.filter(F.col("amount") > 100), "filter (narrow)")
count_exchanges(orders.withColumn("tier", F.when(F.col("amount") > 500, "High").otherwise("Low")),
                "withColumn (narrow)")
count_exchanges(orders.groupBy("customer_id").count(),
                "groupBy + count (wide)")
count_exchanges(orders.join(products, on="product_id", how="inner"),
                "join (wide)")
count_exchanges(orders.orderBy("amount"),
                "orderBy (wide)")
count_exchanges(orders.distinct(),
                "distinct (wide)")

In [ ]:
# Visualize shuffle boundary in explain plan
complex_query = (
    orders
    .join(products, on="product_id", how="inner")   # shuffle 1
    .groupBy("category")                             # shuffle 2
    .agg(F.sum("amount").alias("total"))
    .orderBy(F.col("total").desc())                  # shuffle 3
)

print("Complex query — 3 shuffles:")
complex_query.explain()
print()
print("Each 'Exchange' in the plan = one shuffle boundary = one stage boundary")

## Part 2: Shuffle Metrics in Spark UI

In [ ]:
# Trigger a job to see shuffle metrics
large_df = spark.range(500_000) \
    .withColumn("key",   (F.col("id") % 100).cast("string")) \
    .withColumn("value", F.rand() * 1000)

t0 = time.time()
result = large_df.groupBy("key").agg(F.sum("value"), F.count("*"))
result.count()
print(f"groupBy completed in {time.time()-t0:.2f}s")

print("""
Go to Spark UI → Stages tab and look at the SECOND stage (shuffle read stage):

Metric               What it means
────────────────────  ───────────────────────────────────────────────
Shuffle Read Size    Total bytes read from shuffle files (network)
Shuffle Write Size   Total bytes written to shuffle files (disk)
Shuffle Read Records Number of records read in shuffle read phase
Spill (Memory)       Bytes that were in memory before spilling
Spill (Disk)         Bytes written to disk during spill
Task Duration        Should be similar across tasks (skew if not)
Input Size / Records Input data size per task (skew if one is huge)
────────────────────  ───────────────────────────────────────────────
""")

## Part 3: Reducing Shuffles — Techniques

In [ ]:
# Technique 1: Broadcast small tables — eliminate join shuffle
from pyspark.sql.functions import broadcast

# Without broadcast: SortMergeJoin (2 shuffles — both sides)
t0 = time.time()
no_broadcast = large_df.join(orders, large_df.key == orders.customer_id.cast("string"))
no_broadcast.count()
no_bc_time = time.time() - t0

# With broadcast: BroadcastHashJoin (0 shuffles)
t0 = time.time()
with_broadcast = large_df.join(broadcast(orders),
                                large_df.key == orders.customer_id.cast("string"))
with_broadcast.count()
bc_time = time.time() - t0

print(f"SortMergeJoin (shuffle): {no_bc_time:.3f}s")
print(f"BroadcastHashJoin:       {bc_time:.3f}s")

In [ ]:
# Technique 2: Combine operations to reduce shuffles

# Bad: 2 groupBy operations = 2 shuffles
step1 = orders.groupBy("customer_id").agg(F.sum("amount").alias("total"))  # shuffle 1
step2 = step1.groupBy().agg(F.avg("total").alias("avg_spend"))              # shuffle 2

print("Two separate groupBys (2 shuffles):")
step2.explain()

# Good: combine into one groupBy = 1 shuffle
combined = orders.agg(
    F.avg(F.col("amount")).alias("avg_per_order"),
    F.count("order_id").alias("total_orders")
)
print("Combined agg (0 shuffles):")
combined.explain()

In [ ]:
# Technique 3: Filter before shuffle — reduce data early
# Always filter BEFORE groupBy/join — less data to shuffle

# Bad: shuffle all 500k rows, then filter
df1 = large_df.groupBy("key").agg(F.sum("value").alias("total")) \
              .filter(F.col("total") > 50000)

# Better: can we filter BEFORE? (only if filter is on the pre-shuffle data)
df2 = large_df.filter(F.col("value") > 100) \
              .groupBy("key").agg(F.sum("value").alias("total")) \
              .filter(F.col("total") > 50000)

print(f"Without pre-filter: {large_df.count():,} rows shuffled")
filtered = large_df.filter(F.col("value") > 100)
print(f"With pre-filter:    {filtered.count():,} rows shuffled (~5x less data!)")

In [ ]:
# Technique 4: Pre-sort and bucketed tables (avoids repeated shuffle)
# Write data bucketed by join key → future joins on that key skip shuffle

print("""
Avoiding Shuffles with Bucketed Tables:
─────────────────────────────────────────────────────────────────
# Write once with bucketing:
orders.write.bucketBy(32, 'customer_id').sortBy('customer_id') \\
       .saveAsTable('orders_bucketed')

customers.write.bucketBy(32, 'customer_id').sortBy('customer_id') \\
          .saveAsTable('customers_bucketed')

# Now every join uses pre-sorted buckets — no shuffle!
spark.table('orders_bucketed') \\
     .join(spark.table('customers_bucketed'), on='customer_id')

→ Explain shows NO Exchange nodes for the join!
→ Each executor joins bucket N from orders with bucket N from customers

Use when: same large tables are joined frequently (daily ETL, dashboards)
─────────────────────────────────────────────────────────────────
""")

## Part 4: Shuffle Configuration Knobs

In [ ]:
print("""
Key Shuffle Configuration:
═══════════════════════════════════════════════════════════════
spark.sql.shuffle.partitions          (default: 200)
  Number of partitions for shuffles. Too high → many tiny files.
  Too low → spill. Rule: 2-4× total executor cores.
  AQE auto-tunes this at runtime (Spark 3.0+).

spark.shuffle.file.buffer             (default: 32k)
  Buffer size for shuffle write. Larger = fewer OS write() calls.
  Increase to 64k-128k for large shuffles.

spark.reducer.maxSizeInFlight         (default: 48m)
  Max data fetched simultaneously per reducer task.
  Increase for fast networks (10GbE) to pipeline more fetches.

spark.shuffle.io.maxRetries           (default: 3)
  Retries for failed shuffle fetch (network errors).
  Increase to 6-10 for unstable networks.

spark.shuffle.compress                (default: true)
  Compress shuffle files. Reduces I/O but adds CPU.
  Use LZ4 (default) or Snappy for fast compression.

spark.shuffle.spill.compress          (default: true)
  Compress spill files. Usually leave as true.

spark.sql.adaptive.enabled            (default: true in Spark 3.2+)
  AQE automatically tunes shuffle partitions, join strategies,
  and handles skew at runtime.
═══════════════════════════════════════════════════════════════
""")

## Exercises

1. Count the number of shuffles (Exchange nodes) in a query that: reads orders, joins with products, groups by category, and orders by total revenue descending.
2. Rewrite the query from Exercise 1 using a broadcast hint for products. How many shuffles remain?
3. What is the difference between narrow and wide transformations? Give 3 examples of each.
4. A job has `shuffle.partitions=200` and only processes 100MB of data total. What is the problem and how would you fix it? (Assume AQE is disabled.)
5. Explain why bucketed tables avoid shuffle at join time.

In [ ]:
# Exercise 1 & 2: Count shuffles
q1 = orders.join(products, on="product_id") \
            .groupBy("category") \
            .agg(F.sum("amount").alias("total")) \
            .orderBy(F.col("total").desc())

q2 = orders.join(broadcast(products), on="product_id") \
            .groupBy("category") \
            .agg(F.sum("amount").alias("total")) \
            .orderBy(F.col("total").desc())

count_exchanges(q1, "Without broadcast")
count_exchanges(q2, "With broadcast hint")

print()
print("Plan without broadcast:")
q1.explain()
print("\nPlan with broadcast:")
q2.explain()